In [3]:
pip install statsmodels

Note: you may need to restart the kernel to use updated packages.


In [6]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# Configuration
# ============================================================
INPUT_FILE  = 'analysis_dataset.csv'
OUTPUT_FILE = 'ols_results.csv'

# ============================================================
# Step 1: Load data
# ============================================================
print("Loading analysis dataset...")
df = pd.read_csv(INPUT_FILE, low_memory=False, dtype={'FIPS': str})
print(f"Loaded: {len(df):,} counties")

# ============================================================
# Step 2: Prepare variables
# ============================================================

# Exposure: log(density + 1) — handles zero-inflation and right skew
df['log_density_rad']         = np.log(df['density_radiologist'] + 1)
df['log_density_rad_primary'] = np.log(df['density_radiologist_primary'] + 1)
df['log_density_path']        = np.log(df['density_pathologist'] + 1)

# Covariates
df['rucc_2023']                = pd.to_numeric(df['rucc_2023'], errors='coerce')
df['median_household_income']  = pd.to_numeric(df['median_household_income'], errors='coerce')
df['pct_uninsured_adults']     = pd.to_numeric(df['pct_uninsured_adults'], errors='coerce')

# Income in $10,000 units for interpretable coefficient
df['income_10k'] = df['median_household_income'] / 10_000

# Analysis samples
# Main model requires: outcome + all three covariates
df_main = df[
    df['mortality_rate'].notna() &
    df['rucc_2023'].notna() &
    df['income_10k'].notna() &
    df['pct_uninsured_adults'].notna()
].copy()

df_incd = df[
    df['incidence_rate'].notna() &
    df['rucc_2023'].notna() &
    df['income_10k'].notna() &
    df['pct_uninsured_adults'].notna()
].copy()

print(f"\nMain analysis sample (mortality): {len(df_main):,} counties")
print(f"Incidence sample:                  {len(df_incd):,} counties")
print(f"Dropped (suppressed/missing covariates): {len(df) - len(df_main):,}")
print(f"\nExposure range:  log_density_rad {df_main['log_density_rad'].min():.3f} - {df_main['log_density_rad'].max():.3f}")
print(f"RUCC range:      {df_main['rucc_2023'].min():.0f} - {df_main['rucc_2023'].max():.0f}")
print(f"Income range:    ${df_main['median_household_income'].min():,.0f} - ${df_main['median_household_income'].max():,.0f}")
print(f"Uninsured range: {df_main['pct_uninsured_adults'].min():.1f}% - {df_main['pct_uninsured_adults'].max():.1f}%")

# ============================================================
# Step 3: Model 1 — Baseline (no controls)
# ============================================================
print("\n" + "="*60)
print("MODEL 1: Baseline (no controls)")
print("mortality_rate ~ log(density_radiologist + 1)")
print("="*60)

m1 = smf.ols('mortality_rate ~ log_density_rad', data=df_main).fit(
    cov_type='HC3'
)
print(m1.summary2())

# ============================================================
# Step 4: Model 2 — Main model (all three covariates)
# ============================================================
print("\n" + "="*60)
print("MODEL 2: Main model (RUCC + income + uninsured)")
print("mortality_rate ~ log(density_rad+1) + rucc_2023 + income_10k + pct_uninsured_adults")
print("="*60)

m2 = smf.ols(
    'mortality_rate ~ log_density_rad + rucc_2023 + income_10k + pct_uninsured_adults',
    data=df_main
).fit(cov_type='HC3')
print(m2.summary2())

# ============================================================
# Step 5: Key coefficient summary
# ============================================================
print("\n" + "="*60)
print("KEY RESULTS SUMMARY")
print("="*60)

for label, model in [("Model 1 (no controls)", m1),
                      ("Model 2 (full controls)", m2)]:
    coef  = model.params['log_density_rad']
    se    = model.bse['log_density_rad']
    pval  = model.pvalues['log_density_rad']
    ci_lo = model.conf_int().loc['log_density_rad', 0]
    ci_hi = model.conf_int().loc['log_density_rad', 1]
    r2    = model.rsquared
    n     = int(model.nobs)
    print(f"\n{label}")
    print(f"  β (log_density_rad) = {coef:.4f}")
    print(f"  SE = {se:.4f}  |  p = {pval:.4f}")
    print(f"  95% CI: [{ci_lo:.4f}, {ci_hi:.4f}]")
    print(f"  R² = {r2:.4f}  |  N = {n:,}")

# ============================================================
# Step 6: Sensitivity analyses
# ============================================================
print("\n" + "="*60)
print("SENSITIVITY ANALYSES")
print("="*60)

sensitivity_results = []

# S1: Primary-specialty radiologists only
print("\nS1: Primary-specialty radiologists only")
s1 = smf.ols(
    'mortality_rate ~ log_density_rad_primary + rucc_2023 + income_10k + pct_uninsured_adults',
    data=df_main
).fit(cov_type='HC3')
coef_s1 = s1.params['log_density_rad_primary']
p_s1    = s1.pvalues['log_density_rad_primary']
ci_s1   = s1.conf_int().loc['log_density_rad_primary']
print(f"  β = {coef_s1:.4f}  |  p = {p_s1:.4f}  |  95% CI: [{ci_s1[0]:.4f}, {ci_s1[1]:.4f}]")
sensitivity_results.append({
    'model':'S1_primary_only', 'exposure':'log_density_rad_primary',
    'outcome':'mortality_rate', 'coef':round(coef_s1,4), 'pval':round(p_s1,4),
    'ci_lo':round(ci_s1[0],4), 'ci_hi':round(ci_s1[1],4), 'n':int(s1.nobs),
    'controls':'rucc_2023 + income_10k + pct_uninsured_adults'
})

# S2: Pathologist density as alternative exposure
print("\nS2: Pathologist density (robustness)")
s2 = smf.ols(
    'mortality_rate ~ log_density_path + rucc_2023 + income_10k + pct_uninsured_adults',
    data=df_main
).fit(cov_type='HC3')
coef_s2 = s2.params['log_density_path']
p_s2    = s2.pvalues['log_density_path']
ci_s2   = s2.conf_int().loc['log_density_path']
print(f"  β = {coef_s2:.4f}  |  p = {p_s2:.4f}  |  95% CI: [{ci_s2[0]:.4f}, {ci_s2[1]:.4f}]")
sensitivity_results.append({
    'model':'S2_pathologist', 'exposure':'log_density_path',
    'outcome':'mortality_rate', 'coef':round(coef_s2,4), 'pval':round(p_s2,4),
    'ci_lo':round(ci_s2[0],4), 'ci_hi':round(ci_s2[1],4), 'n':int(s2.nobs),
    'controls':'rucc_2023 + income_10k + pct_uninsured_adults'
})

# S3: Incidence as outcome
print("\nS3: Incidence as outcome")
s3 = smf.ols(
    'incidence_rate ~ log_density_rad + rucc_2023 + income_10k + pct_uninsured_adults',
    data=df_incd
).fit(cov_type='HC3')
coef_s3 = s3.params['log_density_rad']
p_s3    = s3.pvalues['log_density_rad']
ci_s3   = s3.conf_int().loc['log_density_rad']
print(f"  β = {coef_s3:.4f}  |  p = {p_s3:.4f}  |  95% CI: [{ci_s3[0]:.4f}, {ci_s3[1]:.4f}]")
sensitivity_results.append({
    'model':'S3_incidence_outcome', 'exposure':'log_density_rad',
    'outcome':'incidence_rate', 'coef':round(coef_s3,4), 'pval':round(p_s3,4),
    'ci_lo':round(ci_s3[0],4), 'ci_hi':round(ci_s3[1],4), 'n':int(s3.nobs),
    'controls':'rucc_2023 + income_10k + pct_uninsured_adults'
})

# S4: Exclude top 1% density counties (academic medical center outliers)
print("\nS4: Exclude top 1% density outliers (academic medical centers)")
p99 = df_main['density_radiologist'].quantile(0.99)
df_no_outlier = df_main[df_main['density_radiologist'] <= p99].copy()
s4 = smf.ols(
    'mortality_rate ~ log_density_rad + rucc_2023 + income_10k + pct_uninsured_adults',
    data=df_no_outlier
).fit(cov_type='HC3')
coef_s4 = s4.params['log_density_rad']
p_s4    = s4.pvalues['log_density_rad']
ci_s4   = s4.conf_int().loc['log_density_rad']
n_excl  = len(df_main) - len(df_no_outlier)
print(f"  Top 1% threshold: {p99:.2f}/100k  |  Excluded: {n_excl:,} counties")
print(f"  β = {coef_s4:.4f}  |  p = {p_s4:.4f}  |  95% CI: [{ci_s4[0]:.4f}, {ci_s4[1]:.4f}]")
sensitivity_results.append({
    'model':'S4_no_outliers', 'exposure':'log_density_rad',
    'outcome':'mortality_rate', 'coef':round(coef_s4,4), 'pval':round(p_s4,4),
    'ci_lo':round(ci_s4[0],4), 'ci_hi':round(ci_s4[1],4), 'n':int(s4.nobs),
    'controls':'rucc_2023 + income_10k + pct_uninsured_adults'
})

# S5: Rural counties only (RUCC 4-9)
print("\nS5: Rural counties only (RUCC 4-9)")
df_rural = df_main[df_main['rucc_2023'] >= 4].copy()
s5 = smf.ols(
    'mortality_rate ~ log_density_rad + rucc_2023 + income_10k + pct_uninsured_adults',
    data=df_rural
).fit(cov_type='HC3')
coef_s5 = s5.params['log_density_rad']
p_s5    = s5.pvalues['log_density_rad']
ci_s5   = s5.conf_int().loc['log_density_rad']
print(f"  Rural counties: {len(df_rural):,}")
print(f"  β = {coef_s5:.4f}  |  p = {p_s5:.4f}  |  95% CI: [{ci_s5[0]:.4f}, {ci_s5[1]:.4f}]")
sensitivity_results.append({
    'model':'S5_rural_only', 'exposure':'log_density_rad',
    'outcome':'mortality_rate', 'coef':round(coef_s5,4), 'pval':round(p_s5,4),
    'ci_lo':round(ci_s5[0],4), 'ci_hi':round(ci_s5[1],4), 'n':int(s5.nobs),
    'controls':'rucc_2023 + income_10k + pct_uninsured_adults'
})

# S6: RUCC-only controls (drop income + uninsured) — tests robustness to covariate choice
print("\nS6: RUCC-only controls (drop income + uninsured)")
s6 = smf.ols(
    'mortality_rate ~ log_density_rad + rucc_2023',
    data=df_main
).fit(cov_type='HC3')
coef_s6 = s6.params['log_density_rad']
p_s6    = s6.pvalues['log_density_rad']
ci_s6   = s6.conf_int().loc['log_density_rad']
print(f"  β = {coef_s6:.4f}  |  p = {p_s6:.4f}  |  95% CI: [{ci_s6[0]:.4f}, {ci_s6[1]:.4f}]")
sensitivity_results.append({
    'model':'S6_rucc_only', 'exposure':'log_density_rad',
    'outcome':'mortality_rate', 'coef':round(coef_s6,4), 'pval':round(p_s6,4),
    'ci_lo':round(ci_s6[0],4), 'ci_hi':round(ci_s6[1],4), 'n':int(s6.nobs),
    'controls':'rucc_2023'
})

# ============================================================
# Step 7: Save results
# ============================================================
results = []

for label, model, var, outcome, controls in [
    ('M1_baseline', m1, 'log_density_rad', 'mortality_rate', 'none'),
    ('M2_main',     m2, 'log_density_rad', 'mortality_rate',
     'rucc_2023 + income_10k + pct_uninsured_adults'),
]:
    ci = model.conf_int().loc[var]
    results.append({
        'model':    label,
        'exposure': var,
        'outcome':  outcome,
        'coef':     round(model.params[var], 4),
        'se':       round(model.bse[var], 4),
        'pval':     round(model.pvalues[var], 4),
        'ci_lo':    round(ci[0], 4),
        'ci_hi':    round(ci[1], 4),
        'r2':       round(model.rsquared, 4),
        'n':        int(model.nobs),
        'controls': controls
    })

results += sensitivity_results
results_df = pd.DataFrame(results)
results_df.to_csv(OUTPUT_FILE, index=False)

print(f"\n\nAll results saved to: {OUTPUT_FILE}")
print("\n" + "="*60)
print("DONE. Check M2 β before proceeding to script 5 (AI scenarios).")
print("="*60)

Loading analysis dataset...
Loaded: 3,135 counties

Main analysis sample (mortality): 3,073 counties
Incidence sample:                  3,018 counties
Dropped (suppressed/missing covariates): 62

Exposure range:  log_density_rad 0.000 - 6.255
RUCC range:      1 - 9
Income range:    $28,236 - $153,716
Uninsured range: 2.6% - 44.5%

MODEL 1: Baseline (no controls)
mortality_rate ~ log(density_radiologist + 1)
                  Results: Ordinary least squares
Model:              OLS              Adj. R-squared:     0.033     
Dependent Variable: mortality_rate   AIC:                29155.5673
Date:               2026-04-13 12:24 BIC:                29167.6281
No. Observations:   3073             Log-Likelihood:     -14576.   
Df Model:           1                F-statistic:        107.1     
Df Residuals:       3071             Prob (F-statistic): 1.06e-24  
R-squared:          0.034            Scale:              772.11    
---------------------------------------------------------------